# 05 — Step 4: Spatial Attention Pattern Analysis

**Goal**: Visualize what candidate reflection heads actually attend to.

For query tokens in the reflection region, show which key tokens they attend to —
do they look at the corresponding object tokens across the mirror boundary?

In [ ]:
import sys
sys.path.insert(0, '..')

import pickle
import matplotlib.pyplot as plt
from PIL import Image
from pathlib import Path

from scripts.config import (
    EXPERIMENTS_DIR, MIRROR_PROMPTS, SEEDS, MIRROR_DIR, PROMPT_OBJECTS,
    NUM_INFERENCE_STEPS, MAX_TEXT_TOKENS,
)
from scripts.model_loader import load_flux_pipeline
from scripts.roi import get_object_and_reflection_rois
from scripts.generate import generate_with_full_attention
from scripts.attention_extraction import AttentionData
from scripts.visualization import plot_spatial_attention, plot_attention_grid

STEP1_DIR = EXPERIMENTS_DIR / "step1_selectivity"
STEP3_DIR = EXPERIMENTS_DIR / "step3_temporal"
STEP4_DIR = EXPERIMENTS_DIR / "step4_spatial"
STEP4_DIR.mkdir(parents=True, exist_ok=True)

# Load previous results
with open(STEP1_DIR / "step1_results.pkl", "rb") as f:
    step1 = pickle.load(f)
candidates = step1["candidates"]

with open(STEP3_DIR / "step3_results.pkl", "rb") as f:
    step3 = pickle.load(f)
peak_timestep = step3["peak_timestep"]

candidate_blocks = list(set(b for b, h, _ in candidates[:5]))

# Use first mirror prompt for spatial analysis
prompt_idx = 0
seed = SEEDS[0]
obj_name = PROMPT_OBJECTS[prompt_idx]
img_path = MIRROR_DIR / f"prompt{prompt_idx:02d}_seed{seed}.png"
ref_image = Image.open(img_path)
obj_roi, ref_roi = get_object_and_reflection_rois(ref_image, obj_name)

print(f"Top-5 candidates: {[(b, h) for b, h, _ in candidates[:5]]}")
print(f"Peak timestep: {peak_timestep}")
print(f"Blocks to extract: {sorted(candidate_blocks)}")
print(f"CLIPSeg ROI ({obj_name}): obj={obj_roi.size} tokens, ref={ref_roi.size} tokens")

In [ ]:
pipe = load_flux_pipeline(quantize_nf4=True, cpu_offload=True)
print("Model loaded.")

In [ ]:
# Generate with full attention extraction on first mirror prompt
prompt = MIRROR_PROMPTS[0]
seed = SEEDS[0]
print(f"Prompt: {prompt}")
print(f"Seed: {seed}")

attn_data = AttentionData()
image, attn_data = generate_with_full_attention(
    pipe, prompt, seed, obj_roi, ref_roi,
    candidate_blocks=candidate_blocks, attention_data=attn_data,
)

print(f"\nStored {len(attn_data.full_maps)} full attention maps")
print(f"Map keys: {list(attn_data.full_maps.keys())}")

In [ ]:
# Visualize spatial attention for top candidates
# For each candidate: show where reflection-region queries attend

for b, h, s in candidates[:5]:
    key = (b, peak_timestep)
    if key not in attn_data.full_maps:
        # Try other timesteps
        for t in range(NUM_INFERENCE_STEPS):
            if (b, t) in attn_data.full_maps:
                key = (b, t)
                break
        else:
            print(f"No attention map for block {b}, skipping")
            continue

    attn_map = attn_data.full_maps[key]
    txt_len = attn_map.shape[1] - 1024  # total_seq - num_image_tokens

    fig = plot_spatial_attention(
        attn_map, head_idx=h,
        query_indices=ref_roi.token_indices,
        txt_len=txt_len,
        image=image,
        title=f"Block {b}, Head {h} (S={s:.4f}) — Reflection queries",
    )
    fig.savefig(STEP4_DIR / f"spatial_B{b}_H{h}.png", dpi=150, bbox_inches="tight")
    plt.show()

In [ ]:
# Also show: object-region queries attending to reflection region
# (The reverse direction: does the model also attend from object to reflection?)

for b, h, s in candidates[:3]:
    key = (b, peak_timestep)
    if key not in attn_data.full_maps:
        for t in range(NUM_INFERENCE_STEPS):
            if (b, t) in attn_data.full_maps:
                key = (b, t)
                break
        else:
            continue

    attn_map = attn_data.full_maps[key]
    txt_len = attn_map.shape[1] - 1024

    fig = plot_spatial_attention(
        attn_map, head_idx=h,
        query_indices=obj_roi.token_indices,
        txt_len=txt_len,
        image=image,
        title=f"Block {b}, Head {h} — Object queries (reverse)",
    )
    fig.savefig(STEP4_DIR / f"spatial_reverse_B{b}_H{h}.png", dpi=150, bbox_inches="tight")
    plt.show()

In [ ]:
# Full attention matrix visualization for top candidate block
top_block = candidates[0][0]
key = None
for t in range(NUM_INFERENCE_STEPS):
    if (top_block, t) in attn_data.full_maps:
        key = (top_block, t)
        break

if key:
    attn_map = attn_data.full_maps[key]
    txt_len = attn_map.shape[1] - 1024
    top_heads = [h for b, h, _ in candidates if b == top_block]

    fig = plot_attention_grid(
        attn_map, txt_len=txt_len,
        heads_to_show=top_heads[:8],
        title=f"Block {top_block} — Image-to-Image Attention",
    )
    fig.savefig(STEP4_DIR / f"attn_grid_B{top_block}.png", dpi=150, bbox_inches="tight")
    plt.show()

In [ ]:
# Save results
results = {
    "prompt": prompt,
    "seed": seed,
    "candidate_blocks": candidate_blocks,
    "peak_timestep": peak_timestep,
}
with open(STEP4_DIR / "step4_results.pkl", "wb") as f:
    pickle.dump(results, f)

print(f"Results saved to {STEP4_DIR}")
print("\n=== Step 4 Complete ===")
print("Next: Generate HTML report (report/index.html)")